# Transform Circuits Data
 
 1. Read bronze `circuits` table
 2. Keep only the columns required for analytics (Drop `url` column)
 3. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
 4. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
 5. Filter out rows where `circuit_id` is null (business key validation)
 6. Remove duplicate records
 7. Transform values of columns `circuit_name` and `locality` to Title Case
 8. Write the transformed data to silver `circuits` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

#### Step 1 - Read bronze `circuits` table

In [0]:
circuits_df = spark.table(bronze_table)

In [0]:
display(circuits_df)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F

circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("injestion_timestamp"),
    F.col("source_file")
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId":"circuit_id", 
            "circuitName": "circuit_name", 
            "lat": "latitude", 
            "long":"longitude"
        })
)

#### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col('circuit_id').isNotNull()
)

In [0]:
display(circuits_valid_df)

#### Step 6 - Remove duplicate records

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])
display(circuits_distinct_df)

#### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col('circuit_name')))
        .withColumn('locality', F.initcap(F.col('locality')))
)
display(circuits_final_df)

#### Step 8 - Write the transformed data to silver `circuits` table

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))